In [0]:
%sql
 CREATE SCHEMA electiva;
 

In [0]:
import pandas as pd
import requests
from io import StringIO

# Leer datos desde GitHub
url = "https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/202602/DATOS/FT.CSV"
response = requests.get(url)
response.encoding = 'utf-8'

# Crear DataFrame de pandas
df = pd.read_csv(StringIO(response.text),sep=';')

# Mostrar información del DataFrame
print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
display(df.head())
# Convertir a Spark DataFrame
df_spark = spark.createDataFrame(df)

# Guardar como tabla Delta en el catálogo
# Ajusta el nombre de la tabla según tu catálogo y esquema
table_name = "workspace.electiva.ft"
df_spark.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"\nTabla guardada exitosamente como: {table_name}")

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
cgse

In [0]:

#Leyendo la tabla de fuerza de trabajo
ft = spark.read.format("delta").table("workspace.electiva.ft")
ft=ft.toPandas()
ft

In [0]:
cgse[cgse['DIRECTORIO']==8629306]
cgse[cgse['SECUENCIA_P']>3]

Exploracion de modulos y calculos base

**Funciones basicas de pandas**

In [0]:
# Filtrando a nivel de columna
cgse['FEX_C18']

#Aplicando operaciones
cgse['PERIODO'].sum()
cgse['PERIODO'].max()
cgse['PERIODO'].min()
cgse['PERIODO'].mean()
cgse['PERIODO'].std()
cgse['PERIODO'].var()
cgse['PERIODO'].median()
cgse['PERIODO'].mode()

In [0]:
#Primeros n registros de la tabla
cgse.head(20)

In [0]:
cgse.describe()

In [0]:
# Filtrando varias columnas de la tabla nivel de columna
cgse[['PERIODO','FEX_C18','P6090']]
     

In [0]:
#Filtro a nivel de fila, filtrar solo personas donde la variable DPTO=63
cgse[cgse['DPTO'] == 63]
cgse_quindio = cgse[cgse['DPTO'] == 63]

In [0]:
cgse['P3043'].unique()


**1. Poblacion total**

In [0]:
cgse
PT = cgse['FEX_C18'].sum()
PT

Para junio de 2026, la tasa de desocupación del total nacional fue 8,0%
**Como verificamos esto ?**

La tasa de desempleo se calcula mediante la formula:

-> TD=D/PEA*100

D : Cantidad de personas desempleadas
PEA : poblacion economicamente activa
Primer tema a resolver como calcular la cantidad de personas desempleadas ?

In [0]:
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
D

##Cantidad  de desempleados en colombia 2134634.952160001

Segundo tema a resolver , como calcular la PEA ?

In [0]:
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA
## Cantidad de personas en la PEA  en colombia 26593429.254050065
     

**Calculando la tasa de desempleo**

In [0]:
TD = D/PEA*100
print(f'La tasa de desempleo en colombia para junio 2026 es de {TD} % ')

Actividad:

calcular Tasa global de partipacion -> TGP = PEA/PET*100
calcular tasa de ocupacion -> TO = O/PEA100,TO = O/PET100

In [0]:
#Actividad con la tasa global de participacion
PET = ft[ft['PET']==1]['FEX_C18'].sum()
PET
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA
TGP = PEA/PET*100
print(f'La tasa global de participación (TGP) en Colombia para junio 2026 es de {TGP} %')          
#Actividad con la tasa de ocupacion
O = PEA - D
TO = O/PEA*100
TO = O/PET*100
print(f'La tasa de ocupación (TO) en Colombia para junio 2026 es de {TO} %')workspace.electiva.ft

OCUPADOS

In [0]:
ocupados['RAMA2D_R4'].unique()workspace.electiva.ocupados

In [0]:
#Calculando cuantas personas trabajan en el sector de alojamiento y servicios de comida en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==55) | (ocupados['RAMA2D_R4']==56) ]['FEX_C18'].sum()
# En colombia en el sector de alojamiento y servicios de comida trabajan 2.057.354 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de construccion en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==41) | (ocupados['RAMA2D_R4']==42) | (ocupados['RAMA2D_R4']==43) ]['FEX_C18'].sum()
# En colombia en el sector de construccion trabajan  1.678.378  personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de informacion y comunicaciones en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==58) | (ocupados['RAMA2D_R4']==59) | (ocupados['RAMA2D_R4']==60) | (ocupados['RAMA2D_R4']==61) | (ocupados['RAMA2D_R4']==62) | (ocupados['RAMA2D_R4']==63) ]['FEX_C18'].sum()
# En colombia en el sector de informacion y comunicaciones trabajan 386.436 personas a junio de 2026

In [0]:
ocupados[(ocupados['RAMA2D_R4']==10) | (ocupados['RAMA2D_R4']==11) | (ocupados['RAMA2D_R4']==12) | (ocupados['RAMA2D_R4']==13)| (ocupados['RAMA2D_R4']==14)| (ocupados['RAMA2D_R4']==15)| (ocupados['RAMA2D_R4']==16)| (ocupados['RAMA2D_R4']==17)| (ocupados['RAMA2D_R4']==18)| (ocupados['RAMA2D_R4']==19)| (ocupados['RAMA2D_R4']==20)| (ocupados['RAMA2D_R4']==21)| (ocupados['RAMA2D_R4']==22)| (ocupados['RAMA2D_R4']==23)| (ocupados['RAMA2D_R4']==24)| (ocupados['RAMA2D_R4']==25)| (ocupados['RAMA2D_R4']==26)| (ocupados['RAMA2D_R4']==27)| (ocupados['RAMA2D_R4']==28)| (ocupados['RAMA2D_R4']==29)| (ocupados['RAMA2D_R4']==30)| (ocupados['RAMA2D_R4']==31)| (ocupados['RAMA2D_R4']==32)| (ocupados['RAMA2D_R4']==33)]['FEX_C18'].sum()

In [0]:
#Calculando cuantas personas trabajan en el sector de agricultura , ganaderia, caza, silvicultura y pesca en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==1) | (ocupados['RAMA2D_R4']==2) | (ocupados['RAMA2D_R4']==3)]['FEX_C18'].sum()
# En colombia en el sector agricultura , ganaderia, caza, silvicultura y pesca hay 3'280.869 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de transporte y almacenamiento en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==49) | (ocupados['RAMA2D_R4']==50) | (ocupados['RAMA2D_R4']==51) | (ocupados['RAMA2D_R4']==52) | (ocupados['RAMA2D_R4']==53)]['FEX_C18'].sum()
# En colombia en el sector de transporte y almacenamiento trabajan 1.867.131 personas
 

In [0]:
#Calculando cuantas personas trabajan en el sector de actividades inmobiliarias en colombia a junio de 2026
ocupados[ocupados['RAMA2D_R4']==68]['FEX_C18'].sum()
# En colombia en el sector de actividades inmobiliarias trabajan 302.935 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de comercio y reparación de vehiculos en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==45) | (ocupados['RAMA2D_R4']==46) | (ocupados['RAMA2D_R4']==47)]['FEX_C18'].sum()
# En colombia en el sector de comercio y reparación de vehiculos trabajan 4.108.878 personas a junio de 2026

In [0]:
#Entretenimiento
ocupados[(ocupados['RAMA2D_R4'] >= 90) & (ocupados['RAMA2D_R4'] <= 99)]['FEX_C18'].sum()

In [0]:
#Calculando cuantas personas trabajan en Actividades financieras y de seguros  en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==64) | (ocupados['RAMA2D_R4']==65) | (ocupados['RAMA2D_R4']==66) ]['FEX_C18'].sum()

In [0]:

#Suministro de electricidad, gas, agua y gestión de desechos^ incluye la rama de Explotación de minas y canteras
ocupados[ocupados['RAMA2D_R4'].isin([5, 6, 7, 8, 9, 35, 36, 37, 38, 39])]['FEX_C18'].sum()

In [0]:
#Calculando cuantas personas trabajan en Actividades financieras y de seguros  en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==64) | (ocupados['RAMA2D_R4']==65) | (ocupados['RAMA2D_R4']==66) ]['FEX_C18'].sum()

In [0]:
#Calculando cuantas personas trabajan en el sector de actividades profesionales, cientificas y tecnicas
ocupados[(ocupados['RAMA2D_R4']==69) | (ocupados['RAMA2D_R4']==70) | (ocupados['RAMA2D_R4']==71) | (ocupados['RAMA2D_R4']==72)| (ocupados['RAMA2D_R4']==73)| (ocupados['RAMA2D_R4']==74) | (ocupados['RAMA2D_R4']==75)| (ocupados['RAMA2D_R4']==77)| (ocupados['RAMA2D_R4']==78)| (ocupados['RAMA2D_R4']==79)| (ocupados['RAMA2D_R4']==80)| (ocupados['RAMA2D_R4']==81)| (ocupados['RAMA2D_R4']==82)]['FEX_C18'].sum()
# En colombia en el sector de actividades profesionales, cientificas y tecnicas  1.906 personas a junio de 2026 np.float64(1906016.690420165)
     

Calculo de indicadores laborales para el departamento del Quindío Junio 2026 segmentado por genero.
Vamos a responder a la pregunta :
1. Cual es la tasa de desempleo en hombres en el quindio para junio 2026
2. Cual es la tasa de desempleo en mujeres en el quindio para junio 2026

desempleados

In [0]:
cgse[(cgse['DIRECTORIO']==8637743) & (cgse['SECUENCIA_P']==1) & (cgse['ORDEN']==1) ][['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']]
     

Necesitamos traernos la variable p3271 del modulo de cgse, y añadirla al modulo de desempleados para poder calcular tasa de desempleo en hombres y mujeres del quindío

In [0]:
#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')